In [1]:
# load a simple dataset from GCS

from napistu.gcs import downloads
from napistu.network.ng_core import NapistuGraph

napistu_graph_path = downloads.load_public_napistu_asset(
    "test_pathway",
    "../napistu_data",
    subasset = "napistu_graph"
)

napistu_graph = NapistuGraph.from_pickle(napistu_graph_path)

In [2]:
from napistu_torch.load import napistu_graphs

pg_graph = napistu_graphs.napistu_graph_to_pyg_data(napistu_graph)

/Users/sean/Desktop/GITHUB/napistu/lib/napistu-scrapyard/scraps/pytorch/.venv/lib/python3.11/site-packages/torch_geometric/typing.py:124: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: dlopen(/Users/sean/Desktop/GITHUB/napistu/lib/napistu-scrapyard/scraps/pytorch/.venv/lib/python3.11/site-packages/torch_sparse/_hgt_sample_cpu.so, 0x0006): Symbol not found: __ZN2at4_ops11multinomial4callERKNS_6TensorExbNSt3__18optionalINS_9GeneratorEEE
  Referenced from: <B6FBC735-D615-3F24-96FC-D24E4BE1FC5B> /Users/sean/Desktop/GITHUB/napistu/lib/napistu-scrapyard/scraps/pytorch/.venv/lib/python3.11/site-packages/torch_sparse/_hgt_sample_cpu.so
  Expected in:     <B6BD92AE-4D03-3F92-9E03-2E2594A12866> /Users/sean/Desktop/GITHUB/napistu/lib/napistu-scrapyard/scraps/pytorch/.venv/lib/python3.11/site-packages/torch/lib/libtorch_cpu.dylib
  warnings.warn(f"An issue occurred while importing 'torch-sparse'. "


## Ideas for problems

### High value; long-term

- cross-source edge prediction - can we predict IntAct edges based on OmniPath? How account for overlapping data informing sources.
- can we predict perturbation effects? From a source vertex, can we predict the set of vertices that will be affected?
   - the reverse of the above: can we predict a perturbation from its effects?
- hierarchical biological representations. can we improve community detection (pathway characterization). This could involve creating graphs at varying coarseness.

   - Level 0: Individual molecular interactions (from IntAct, STRING physical)
   - Level 1: Functional modules (STRING functional, local pathway subgraphs)
   - Level 2: Pathway-level (Reactome pathways, KEGG modules)
   - Level 3: Pathway crosstalk (consensus model connections between pathways)
   - Level 4: Biological processes (GO-slim equivalent)

### Benchmarks

- how does a Napistu graph compared to a simple STRING undirected graph?
- what sources are most important for prediction?

### Easy
- community detection - can we take the consensus Reactome model and predict the original submodels?


### Random thoughts
- can GNN's learn conservation of mass? (if some substrates and products are missing, can they be predicted?)


In [ ]:



# can GNN's learn conservation of mass? (if some substrates and products are missing, can they be predicted?)

# can we predict perturbation effects? From a source vertex, can we predict the set of vertices that will be affected?

# the reverse of the above: can we predict a perturbation from its effects?

# community detection - can we take the consensus Reactome model and predict the original submodels?

# cross-source edge prediction - can we predict IntAct edges based on OmniPath? How account for overlapping data informing sources.

# how does a Napistu graph compared to a simple STRING undirected graph?

# what sources are most important for prediction?

napistu_graph

In [ ]:
# load the genome-scale graph
# filter to just STRING intearctions and compare with full graph

# predict GO category membership (hold out 20% of the vertices in each category)

# what edge features exist? We should properly encode the source information.

In [8]:
from napistu.gcs import downloads
from napistu.network.ng_core import NapistuGraph
from napistu.sbml_dfs_core import SBML_dfs



In [3]:
napistu_graph_path = downloads.load_public_napistu_asset(
    "human_consensus",
    "../napistu_data",
    subasset = "napistu_graph"
)

napistu_graph = NapistuGraph.from_pickle(napistu_graph_path)

In [9]:
sbml_dfs_path = downloads.load_public_napistu_asset(
    "human_consensus",
    "../napistu_data",
    subasset = "sbml_dfs"
)

sbml_dfs = SBML_dfs.from_pickle(sbml_dfs_path)

In [17]:
reaction_sources = sbml_dfs.get_source_total_counts("reactions")

INFO:napistu.sbml_dfs_core:Excluding reactions which are all interactors from reaction source counts


In [ ]:
# TO DO - https://github.com/napistu/napistu-py/issues/188
# add reactions -> source function so we can numerically encode reaction and species provenance.

pathway_id
uncompartmentalization                                                          32347
napistu_data/human_consensus/cache/reactome/uncompartmentalized_reactome.pkl    15185
napistu_data/human_consensus/cache/uncompartmentalized_trrust.pkl                8427
napistu_data/human_consensus/cache/uncompartmentalized_bigg.pkl                  8055
R-HSA-162582                                                                     2552
                                                                                ...  
R-HSA-9645722                                                                       1
R-HSA-5083627                                                                       1
R-HSA-9646303                                                                       1
R-HSA-9646304                                                                       1
R-HSA-5619081                                                                       1
Name: total_counts, Length: 2788, dtype: in

In [ ]:
pg_graph = napistu_graphs.napistu_graph_to_pyg_data(napistu_graph, verbose = True)

In [5]:
from napistu_torch.load import transforms
from napistu_torch.load import napistu_graphs

import logging
logger = logging.getLogger(__name__)

In [6]:
vertex_df, edge_df = napistu_graph.to_pandas_dfs()

# 2. Encode node and edge data in numpy arrays
vertex_features, _ = transforms.encode_dataframe(
    vertex_df, napistu_graphs.VERTEX_DEFAULT_TRANSFORMS, verbose=True
)

edge_features, _ = transforms.encode_dataframe(
    edge_df, napistu_graphs.EDGE_DEFAULT_TRANSFORMS, verbose=True
)

INFO:napistu_torch.load.transforms:cat (OneHotEncoder): ['node_type', 'species_type']
INFO:napistu_torch.load.transforms:cat (OneHotEncoder): ['direction', 'sbo_term']
INFO:napistu_torch.load.transforms:num (StandardScaler): ['stoichiometry', 'weight', 'upstream_weight']
INFO:napistu_torch.load.transforms:bool (passthrough): ['r_isreversible']


In [7]:
edge_df

,source,target,from,to,r_id,sbo_term,stoichiometry,sc_Source,species_type,r_isreversible,direction,string_wt,weight,upstream_weight,source_wt
0,38776,2,R00000000,SC00000002,R00000000,SBO:0000011,1.0,<napistu.source.Source object at 0x35063fb10>,metabolite,False,forward,NaN,0.5,0.5,1
1,38777,3,R00000001,SC00000003,R00000001,SBO:0000011,1.0,<napistu.source.Source object at 0x35065d090>,metabolite,False,forward,NaN,0.5,0.5,1
2,38778,0,R00000002,SC00000000,R00000002,SBO:0000011,1.0,<napistu.source.Source object at 0x35063c650>,metabolite,False,forward,NaN,0.5,0.5,1
3,38779,15016,R00000003,SC00015016,R00000003,SBO:0000011,1.0,<napistu.source.Source object at 0x37edef190>,protein,False,forward,NaN,0.5,0.5,1
4,38780,42,R00000004,SC00000042,R00000004,SBO:0000011,1.0,<napistu.source.Source object at 0x350753dd0>,protein,False,forward,NaN,0.5,0.5,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7893752,38772,64529,SC00038772,R02125243,R02125243,SBO:0000459,0.0,<napistu.source.Source object at 0x39afd32d0>,protein,False,forward,NaN,0.5,0.5,1
7893753,38772,64531,SC00038772,R02125577,R02125577,SBO:0000459,0.0,<napistu.source.Source object at 0x39afd32d0>,protein,False,forward,NaN,0.5,0.5,1
7893754,38772,64533,SC00038772,R02125739,R02125739,SBO:0000459,0.0,<napistu.source.Source object at 0x39afd32d0>,protein,False,forward,NaN,0.5,0.5,1
7893755,38773,65031,SC00038773,R03503152,R03503152,SBO:0000019,0.0,<napistu.source.Source object at 0x39afd3e50>,protein,False,forward,NaN,0.5,0.5,1


In [ ]:
vertices, edges = napistu_graph.to_pandas_dfs()

In [ ]:
vertex_features